### GRM Clean Script ###

In [ ]:
import json

with open("principles_clean.json", "r") as f:
    data = json.load(f)

for i in range(len(data["all_principles"])):
    data["all_principles"][i] = data["all_principles"][i].strip(" *")

no_principles_extracted = 0
for entry in data["raw_observations"]:
    if len(entry["principles"]) == 0:
        no_principles_extracted += 1

with open("principles_clean.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

all_principles = list(set(data["all_principles"]))
all_raw_principles = data["all_principles"]

print(f"{no_principles_extracted} Q&Rs had no principles extracted.")
print(f"{len(all_principles)} unique principles.")
print(f"{len(all_raw_principles)} principles extracted from 200 Q&Rs")
print(all_principles)

34 Q&Rs had no principles extracted.
11 unique principles.
673 principles extracted from 200 Q&Rs
['Level of Detail', 'Error Handling', 'Usefulness', 'Nobel Prize Win', 'British-born Requirement', 'Language Compatibility', 'Functionality Preservation', 'Instruction Adherence', 'Relevance', 'Accuracy of Information', 'Readability and Maintainability']


# Definition Extraction

In [2]:
import re
import json

# Need to remove points and number percentages from weight analysis.
def check_invalid(specific_criteria):
    keywords = ["response 1", "response 2", "weight", "%", "both responses"]
    if len(specific_criteria) == 0:
        return True
    for criterion in specific_criteria:
        if any(word in criterion["definition"].lower() for word in keywords):
             return True
        if any(word in criterion["name"].lower() for word in keywords):
             return True
    return False

# Filters Criteria Definitions
def extract_criteria_definitions(text):
    # Extract section
    criteria_block = re.search(
        r'(?s)Specific Criteria:(.*?)(?=Analysis:)',
        text
    )

    if not criteria_block:
        return []
    criteria_block = criteria_block.group(1)

    # Extract structured entries
    matches = re.findall(
        r'(\d+)\.\s*(.*?):\s*(.*?)(?=\n\d+\.|\Z)',
        criteria_block,
        flags=re.S
    )

    # Remove points from definition and clean up names
    parsed = [
        {
            "name": name.strip(" *"),
            "definition": re.sub(r"\d+\s+points\s*-\s*", "", definition, flags=re.IGNORECASE).strip()
        }
        for _, name, definition in matches
    ]
    return parsed

invalid_count = 0

with open("new_runs_v2/principles.json", "r") as f:
    data = json.load(f)

for i, entry in enumerate(data["raw_observations"]):
    new_criteria = extract_criteria_definitions(entry["judgement"])
    data["raw_observations"][i]["invalid"] = False
    if check_invalid(new_criteria):
        data["raw_observations"][i]["invalid"] = True
        invalid_count += 1
    data["raw_observations"][i]["specific_criteria"] = new_criteria

with open("new_runs_v2/principles_clean.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Number of Q&Rs with no criteria extracted: {invalid_count} / {len(data['raw_observations'])}")

valid_items = [obj for obj in data["raw_observations"] if not obj["invalid"]]

with open("new_runs_v2/principles_valid.json", "w", encoding="utf-8") as f:
    json.dump(valid_items, f, ensure_ascii=False, indent=2)


Number of Q&Rs with no criteria extracted: 99 / 200


In [3]:
import re
import json

# Need to remove points and number percentages from weight analysis.
def check_invalid(specific_criteria):
    keywords = ["response 1", "response 2", "both responses"]
    if len(specific_criteria) == 0:
        return True
    for criterion in specific_criteria:
        if any(word in criterion["definition"].lower() for word in keywords):
             return True
        if any(word in criterion["name"].lower() for word in keywords):
             return True
    return False

# Filters Criteria Definitions
def extract_criteria_definitions(text):
    # Extract section
    criteria_block = re.search(
        r'(?s)Specific Criteria:(.*?)(?=Analysis:)',
        text
    )

    if not criteria_block:
        return []
    criteria_block = criteria_block.group(1)

    # Extract structured entries
    matches = re.findall(
        r'(\d+)\.\s*(.*?):\s*(.*?)(?=\n\d+\.|\Z)',
        criteria_block,
        flags=re.S
    )

    # Remove points from definition and clean up names
    parsed = [
        {
            "name": name.strip(" *"),
            "definition": re.sub(r"\d+\s+points\s*-\s*", "", definition, flags=re.IGNORECASE).strip()
        }
        for _, name, definition in matches
    ]
    return parsed

invalid_count = 0

with open("new_runs_v2/principles.json", "r") as f:
    data = json.load(f)

for i, entry in enumerate(data["raw_observations"]):
    new_criteria = extract_criteria_definitions(entry["judgement"])
    data["raw_observations"][i]["invalid"] = False
    if check_invalid(new_criteria):
        data["raw_observations"][i]["invalid"] = True
        invalid_count += 1
    data["raw_observations"][i]["specific_criteria"] = new_criteria

print(f"Number of Q&Rs with no criteria extracted: {invalid_count} / {len(data['raw_observations'])}")

Number of Q&Rs with no criteria extracted: 45 / 200


In [7]:
import json

with open("interp_runs/interpreted_runs.json", "r") as f:
    data = json.load(f)

for i in range(len(data)):
    # Get rid of thinking part in score
    content = data[i]["score"][1]["content"]

    if "</think>" in content:
        thinking, response = content.split("</think>", 1)

        # Clean thinking section
        thinking = thinking.replace("<think>", "").strip()

        try:
            data[i]["thinking"] = thinking
            data[i]["score"] = int(response.strip())
        except ValueError:
            data[i]["thinking"] = thinking
            data[i]["score"] = None
    else:
        data[i]["thinking"] = None
        data[i]["score"] = None

with open("interp_runs/interpreted_runs_clean.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

non_zero_data = [item for item in data if item.get("score") not in (None, 0)]

with open("interp_runs/interpreted_runs_non_zero.json", "w", encoding="utf-8") as f:
    json.dump(non_zero_data, f, ensure_ascii=False, indent=2)

print(f"Entries with non-zero scores: {len(non_zero_data)}")
print(f"Total entries: {len(data)}")


Entries with non-zero scores: 23
Total entries: 200
